In [ ]:
from importlib import reload
from os.path import join
from collections import defaultdict
import sys
import os
import shutil
import numpy as np
from matplotlib import pyplot
import cv2
from functools import partial
cv2.setNumThreads(1)
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from typing import Any

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as tf
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms

In [ ]:
from trackertraincode.datasets.batch import Batch, Tag
from trackertraincode.datasets.dshdf5pose import Hdf5PoseDataset
import trackertraincode.datatransformation as dtr
import trackertraincode.utils as utils
import trackertraincode.neuralnets.models as models
import trackertraincode.pipelines as pipelines
from trackertraincode.neuralnets.losses import LocalizerBoxLoss, LocalizerProbLoss
import trackertraincode.vis as vis
import trackertraincode.train as train

In [ ]:
%matplotlib ipympl

In [ ]:
net = models.LocalizerNet()
net.cuda()

In [ ]:
# Adjust as needed
path = join(os.environ['DATADIR'],'widerfacessingle.h5')

h, w = net.input_resolution

def make_sample_transform(train : bool):
    crop_resize = (
        dtr.batch.RandomFocusRoi((w,h), 0., 1., roi_variable=None) if train else
        dtr.batch.FocusRoi((w, h), 1., roi_variable=None)
    )
    transforms = [
        dtr.batch.offset_points_by_half_pixel,  # For when pixels are considered cell centered
        crop_resize,
        partial(dtr.batch.horizontal_flip_and_rot_90, 0),
        partial(dtr.batch.normalize_batch),
    ]
    return transforms

train_batch_processing = [
    lambda batch: batch.to("cuda"),
    dtr.batch.KorniaImageDistortions(
        dtr.batch.RandomEqualize(p=0.2),
        dtr.batch.RandomPosterize((4.0, 6.0), p=0.01),
        dtr.batch.RandomGamma((0.5, 2.0), p=0.2),
        dtr.batch.RandomContrast((0.7, 1.5), p=0.2),
        dtr.batch.RandomBrightness((0.7, 1.5), p=0.2),
        dtr.batch.RandomGaussianBlur(
            p=0.1, kernel_size=(5, 5), sigma=(1.5, 1.5)
        ),
        random_apply=4,
    ),
    dtr.batch.KorniaImageDistortions(
        dtr.batch.RandomGaussianNoise(std=4.0 / 255.0, p=0.25),
        dtr.batch.RandomGaussianNoise(std=16.0 / 255.0, p=0.25**2),
        dtr.batch.RandomGaussianNoise(std=32.0 / 255.0, p=0.25**3),
        dtr.batch.RandomGaussianNoise(std=64.0 / 255.0, p=0.25**4),
        dtr.batch.OnlyClip(p=1.0),
    ),
    dtr.batch.whiten_batch
]

trainset, testset = pipelines.make_widerface_datasets()
trainset = dtr.TransformedDataset(trainset, transforms.Compose(make_sample_transform(train=True)))
testset = dtr.TransformedDataset(testset, transforms.Compose(make_sample_transform(train=False)))

In [ ]:
train_loader = dtr.PostprocessingLoader(trainset, 
                          batch_size=256,
                          num_workers=utils.num_workers(),
                          worker_init_fn=pipelines.seed_worker,
                          collate_fn=Batch.collate,
                          shuffle=True,
                          postprocess = transforms.Compose(train_batch_processing))
test_loader = dtr.PostprocessingLoader(testset, 
                          batch_size=256,
                          num_workers=utils.num_workers(),
                          worker_init_fn=pipelines.seed_worker,
                          collate_fn=Batch.collate,
                          shuffle=False, 
                          postprocess = transforms.Compose([
                              lambda batch: batch.to("cuda"),
                              dtr.batch.whiten_batch
                          ]))

In [ ]:
def iterate_loader(loader):
    for batch in loader:
        batch['image'] += 0.5
        batch = dtr.batch.unnormalize_batch(batch)
        batch = batch.to("cpu")
        batch =  dtr.batch.to_numpy(batch)
        yield from Batch.undo_collate(batch)

vis.matplotlib_plot_iterable(iterate_loader(train_loader), vis.draw_dataset_sample)

In [ ]:
vis.matplotlib_plot_iterable(iterate_loader(test_loader), vis.draw_dataset_sample)

In [ ]:
class LocalizerTrainingModule(pl.LightningModule):
    def __init__(self, model : models.LocalizerNet):
        super().__init__()
        self._model = model
        self._losses = train.CriterionGroup([
            train.Criterion('bce', LocalizerProbLoss(), 0.2),
            train.Criterion('box', LocalizerBoxLoss(), 0.8)
        ], "localizer_", 1.0)

    def configure_optimizers(self):
        n_epochs = self.trainer.max_epochs
        assert isinstance(n_epochs, int)
        optimizer = optim.AdamW(net.parameters(), lr=1.e-3, weight_decay=1.e-4)
        scheduler = train.LinearUpThenSteps(optimizer, 5, 0.1, [n_epochs // 2])
        return [optimizer], [scheduler] #{"optimizer": optimizer, "lr_scheduler": scheduler}

    def training_step(self, batch: Batch, batch_idx):
        preds = self._model(batch['image'])
        lossvals = self._losses.evaluate(preds, batch, batch_idx)
        loss_sum = torch.stack([c.val * c.weight for c in lossvals]).mean()
        loss_val_by_name = { c.name:c.val.detach().cpu() for c in lossvals }
        self.log(
            "loss",
            loss_sum,
            on_epoch=True,
            prog_bar=True,
            batch_size=batch.meta.batchsize,
        )
        # Contents of "mt_losses" is plotted.
        return {"loss": loss_sum, "mt_losses": loss_val_by_name}

    def validation_step(
        self, batch: Batch, batch_idx: int
    ) -> list[train.LossVal]:
        pred = self._model(batch["image"])
        values = self._losses.evaluate(pred, batch, batch_idx)
        val_loss = torch.stack([(lv.val * lv.weight) for lv in values]).sum()
        self.log("val_loss", val_loss, on_epoch=True, batch_size=batch.meta.batchsize)
        return values

    @property
    def model(self):
        return self._model

In [ ]:
num_epochs = 200
batchsize = 256
localizer_train_module = LocalizerTrainingModule(net)
progress_cb = train.SimpleProgressBar(batchsize)
visu_cb = train.MetricsGraphing()
swa_callback = train.SwaCallback(start_epoch=num_epochs * 3 // 4)
callbacks = [visu_cb, progress_cb, swa_callback]
torch.set_float32_matmul_precision("high")

In [ ]:
model_out_dir = "../model_files/current/localizer-retrain"
os.makedirs(model_out_dir, exist_ok=True)
print (f"Train set size = {len(trainset)}")

trainer = pl.Trainer(
    fast_dev_run=False,
    #gradient_clip_val=1.0,
    #gradient_clip_algorithm="norm",
    default_root_dir=model_out_dir,
    callbacks=callbacks,
    enable_checkpointing=False,
    max_epochs=num_epochs,
    log_every_n_steps=10,
    logger=False,
    enable_progress_bar=False,
    precision="16-mixed",
)

trainer.fit(model=localizer_train_module, train_dataloaders=train_loader, val_dataloaders=test_loader)
visu_cb.close()